In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model(
    "gemini-3-flash-preview", model_provider="google-genai", api_key=os.getenv("GOOGLE_API_KEY")
)
standard_model = init_chat_model(
    "gemini-3.1-flash-lite", model_provider="google-genai", api_key=os.getenv("GOOGLE_API_KEY")
)


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Oh! Good catch—thanks for the reminder! I honestly got distracted by the mountain of filing on my desk this morning. \n\nI’m heading to the breakroom to grab some water for my coffee right now, so I’ll swing by and take care of the plant immediately. I think it’s been looking a little droopy lately, so I'll make sure it gets a good drink.\n\nIs there anything else you need me to grab while I’m up, or any other tasks you need help with?", 'extras': {'signature': 'EnEKbwFpFH0Ta9g5huQUCrsmeBh6opzRJndYfjqjZ35tB98iCpG/OC4JbDyX6djvD1fF9NoeXERW9PbyJ2ZyQoUKcF9yPU/0mc/tu8oflbgRx1Lcl005pAk36dQxfQTCH35MDk41F8t8SyejTViSNNlpJA=='}}]


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I’ve been keeping an eye on that! Usually, we won’t need to repot it until the roots start poking through the drainage holes at the bottom or if the plant starts looking a bit too big for its base. \n\nBased on its current growth rate, I’d guess we’re probably good for another year or so, but I’ll check the bottom of the pot next time I water it just to make sure it\'s not getting root-bound. Do you want me to add "larger pot" to the office supply request list for next quarter just in case?', 'extras': {'signature': 'EqYNCqMNAWkUfRNxY3tefE9+ktcToQB1CVCrAVUfCcvCFMMnL/YqrPadTzSis8dmdtBNgXH3bVQW5b+AoYlKrong0nQ9s0arCJADIU1YB6xqZc4F/pZhsH1yUvfkiTUYvpWHoV9QIzA3xqxC8NjvOQVzwhFqsyQN2hg66y9hkjjvk6eehoIhI0GcSIWCUNDtI6UaMXjpigXl6+jBxT/GnEBMTJ4p+HaiYr////X276uP2DbZE4wQu6lf3I80Jmy1neeXq3sXDXTvHx1oHCZSvei/Bw8sj/L8gyeL7gTecbvdlAw4AFh5inFxcjsF/oO9Xt1Re4025/8G9myyVYHeN3DChesl4hfVWIRZ6ZE69xh1LcUvuNkDNWUD2L97ZWgtPsK6S9Iuc9nIKtP/jUAPnYcwGEiF9acwcr/ldJ6f7m/HLpgAP9QAMy4xnzk0dF7RbT

In [8]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3-flash-preview
